# Verifikasi jaringan distribusi PT XYZ

Notebook menjalankan data dan prosedur yang dipakai pada tabel GitHub. Contoh bawaan menjalankan satu paket untuk P3 pada Low, Medium, dan High. Untuk benchmark lengkap, ubah `mode` ke `"lengkap"`. Setiap model memakai satu thread HiGHS dan diulang tiga kali setelah pemanasan.

P3, P4, P7, dan P8 menguji 25 batas emisi. Grafik menampilkan pasangan biaya–emisi yang berbeda. Pada P2, P4, P6, dan P8, emisi yang ditampilkan adalah emisi harapan. Python 3.12 digunakan untuk hasil acuan.

In [ ]:
from pathlib import Path
import importlib.metadata
import os
import subprocess
import sys

os.environ["OPENBLAS_NUM_THREADS"] = "1"
folder_repo = "Lampiran-Model-Matematika-Tesis"
kandidat = [Path.cwd(), Path.cwd().parent, Path.cwd() / folder_repo]
akar = next((p for p in kandidat if (p / "math_model_acuan/benchmark.py").is_file()), None)
if akar is None:
    subprocess.run(["git", "clone", "https://github.com/kiittkiidd/Lampiran-Model-Matematika-Tesis.git", folder_repo], check=True)
    akar = Path.cwd() / folder_repo

deps = dict(line.split("==") for line in (akar / "requirements.txt").read_text().splitlines() if line.strip())
terpasang = {p.metadata["Name"].lower(): p.version for p in importlib.metadata.distributions()}
if any(terpasang.get(name) != versi for name, versi in deps.items()):
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(akar / "requirements.txt")], check=True)

In [ ]:
import json
from tempfile import TemporaryDirectory

mode = "contoh"
profil = "P3"
paket = 10101
if mode not in {"contoh", "lengkap"}:
    raise ValueError("Pilih mode contoh atau lengkap.")

temp = TemporaryDirectory()
output = Path(temp.name) / "benchmark.jsonl"
cmd = [sys.executable, str(akar / "math_model_acuan/benchmark.py"), "--output", str(output)]
if mode == "contoh":
    cmd += ["--paket", str(paket), "--profil", profil]
subprocess.run(cmd, cwd=akar, check=True)
hasil = [json.loads(line) for line in output.read_text().splitlines()]
subprocess.run([sys.executable, str(akar / "math_model_acuan/audit_wcc.py")], cwd=akar, check=True)

In [ ]:
import matplotlib.pyplot as plt

label = dict(L="Low", M="Medium", H="High")
warna = dict(L="#17714e", M="#af3344", H="#22262c")
pilihan = [r for r in hasil if r["profile"] == profil and r["seed"] == paket]
for r in pilihan:
    print(f'{profil} {label[r["level"]]} | {r["median_ms"]:.1f} ms | {r["lp"]} iterasi LP | {r["calls"]} panggilan')

if profil in {"P3", "P4", "P7", "P8"}:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for r in pilihan:
        titik = r["frontier"]
        ax.scatter([p["emission"] for p in titik], [p["cost"] for p in titik],
                   label=label[r["level"]], color=warna[r["level"]], alpha=.8)
    ax.set(xlabel="Emisi (kg CO₂e)" if profil in {"P3","P7"} else "Emisi harapan (kg CO₂e)",
           ylabel="Biaya (satuan biaya)" if profil in {"P3","P7"} else "Biaya harapan (satuan biaya)",
           title=f"Sampel Pareto · {profil} · paket {paket}")
    ax.legend()
    ax.grid(alpha=.2)
    plt.show()